# Kinetic neighborhood

Equal-temperature reservoirs provide an equilibrium reference; unequal
reservoirs drive heat transport and density redistribution. Both runs
start independently from the same uniform distribution. Only the kinetic
model is simulated here.

In [ ]:
from pathlib import Path
import sys

import numpy as np
from matplotlib import pyplot as plt

N_STEPS = 2000
REPORT_EVERY = 100
DT = 0.04  # Time per Markov step, as in the source demonstration.
EARLY_SAMPLES = min(120, N_STEPS // 2 + 1)
BATH_BETAS = {
    "Equilibrium": (1.5, 1.5),
    "Nonequilibrium": (1.1, 1.9),
}

## Calibration and boundary conditions

The calibration data `examples/calibration.npz` supplies `multiplicities[ell, i, m, j]` and
`degeneracies[ell, i]`, where `(m, j)` is the source and `(ell, i)` is the
destination. The class `Model` combines the
multiplicities, destination degeneracies, and Boltzmann factors when
constructing each transition kernel.

The calibration data determine the number of sites and energy levels: 25 sites and 5 levels. The model
therefore uses $\epsilon_i=i^2/2$ for $i=1,\ldots,5$:
`[0.5, 2, 4.5, 8, 12.5]`.

Only the first and last sites are thermostatted. Their inverse
temperatures are $(1.5,1.5)$ or $(1.1,1.9)$; the interior inverse
temperatures are solved from local energy balance at every step.

In [ ]:
candidates = [Path("calibration.npz"), Path("examples/calibration.npz")]
calibration_path = next((path for path in candidates if path.is_file()), None)
if calibration_path is None:
    raise FileNotFoundError(
        "Run this notebook from the repository root or examples/; "
        "examples/calibration.npz is required."
    )

with np.load(calibration_path, allow_pickle=False) as calibration:
    multiplicities = calibration["multiplicities"].copy()
    degeneracies = calibration["degeneracies"].copy()
    energies = calibration["energies"].copy() if "energies" in calibration else None

assert degeneracies.ndim == 2
nsites, nlevels = degeneracies.shape
assert (nsites, nlevels) == (25, 5), "Expected the supplied calibration."
assert multiplicities.shape == (nsites, nlevels, nsites, nlevels)
assert np.all(np.isfinite(multiplicities)) and np.all(multiplicities >= 0)
assert np.all(np.isfinite(degeneracies)) and np.all(degeneracies > 0)
assert np.all(multiplicities.sum(axis=(0, 1)) > 0)

dx = 1.0 / nsites
sites = np.arange(1, nsites + 1)
print(f"Calibration: {calibration_path}; {nsites} sites, {nlevels} levels")
print(f"Steps per case: {N_STEPS:,}; final time: {DT * N_STEPS:g}")

## Simulation and Kullback-Leibler diagnostic

Simulation start with $p_{\ell i}=1/(25\times5)$.

Both, equilibrium and nonequilibrium, cases run in separate spawned processes using the `npy` kernel.
The shared `_parallel_simulation.py` module keeps workers importable.

To compare the conditional energy distribution
$p(i\mid\ell)=p_{\ell i}/\sum_j p_{\ell j}$ with the local Boltzmann
distribution $q(i\mid\ell)\propto g_{\ell i}e^{-\beta_\ell\epsilon_i}$, we evaluate the Kullback-Leibler divergence

$$D_\ell=\sum_i p(i\mid\ell)
\log\frac{p(i\mid\ell)}{q(i\mid\ell)}.$$

**NB!**: simulations may take some time...

In [ ]:
# Support launching from either the repository root or examples/.
helper_dir = next(
    path.resolve()
    for path in (Path("."), Path("examples"))
    if (path / "_parallel_simulation.py").is_file()
)
for path in (helper_dir.parent, helper_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from _parallel_simulation import run_cases  # noqa: E402 (path setup above)

results = run_cases(
    BATH_BETAS,
    nsites,
    nlevels,
    N_STEPS,
    REPORT_EVERY,
    model_options={
        "connectivity": multiplicities,
        "degeneracy": degeneracies,
        "energies": energies,
    },
)

## Temperature and density

Temperature is $T_\ell=1/\beta_\ell$ in reduced units, whereas density is evaluated as
$\rho_\ell=\sum_i p_{\ell i}/\Delta x$ and normalized so that $\sum_\ell\rho_\ell\Delta x=1$.

In [ ]:
ORANGE = "#F97316"
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.axisbelow": True,
    }
)


def style_axis(ax):
    ax.grid(True, color="#E2E8F0", linewidth=0.6)


fig_profiles, axes = plt.subplots(
    2, 2, figsize=(9, 6), sharex=True, sharey="col", layout="constrained"
)
for row, (name, result) in enumerate(results.items()):
    state = result["model"].state
    assert np.all(state.betas > 0)
    density = state.site_occupancies() / dx
    np.testing.assert_allclose(density.sum() * dx, 1.0, atol=1e-12, rtol=0)
    profiles = (1.0 / state.betas, density)
    for col, (ax, profile) in enumerate(zip(axes[row], profiles, strict=True)):
        ax.plot(
            sites,
            profile,
            color=ORANGE,
            marker="o",
            markerfacecolor="none",
            markersize=5,
            linestyle="none",
            label="Kinetic neighborhood",
        )
        quantity = "Temperature" if col == 0 else "Density"
        ax.set_title(f"{name} · {quantity}")
        ax.set_ylabel(r"$T_\ell$" if col == 0 else r"$\rho_\ell$")
        ax.set_xticks([1, 7, 13, 19, 25])
        ax.set_xlim(0.5, nsites + 0.5)
        ax.set_ylim(*((0.5, 1.0) if col == 0 else (0.8, 1.3)))
        style_axis(ax)
        if row == 1:
            ax.set_xlabel(r"Site, $\ell$")

fig_profiles.suptitle(f"Kinetic neighborhood · after {N_STEPS:,} steps")
plt.show()

## Nonequilibrium relaxation toward local Boltzmann distributions

The orange band spans the minimum and maximum KL divergence across all
25 sites, including the reservoirs; it is a **spatial range**, not an
uncertainty interval. The left panel shows the full trajectory and the
right shows the first 51 samples (steps 0–50) to resolve early relaxation
within this shorter run. The early window is capped at the original 120
samples for longer runs. Time is $t=0.04\,n$ for step $n$; only the vertical axis is
logarithmic so that the initial sample at $t=0$ remains visible.

In [ ]:
kl_ne = results["Nonequilibrium"]["kl"]
time = DT * np.arange(N_STEPS + 1)
lower = np.nanmin(kl_ne, axis=1)
upper = np.nanmax(kl_ne, axis=1)
plotted_lower = np.ma.masked_less_equal(lower, 0)
plotted_upper = np.ma.masked_less_equal(upper, 0)

fig_kl, axes_kl = plt.subplots(
    1, 2, figsize=(9, 3.4), sharey=True, layout="constrained"
)
windows = [slice(None), slice(0, EARLY_SAMPLES)]
titles = ["Full trajectory", f"First {EARLY_SAMPLES} samples"]
for ax, window, title in zip(axes_kl, windows, titles, strict=True):
    ax.fill_between(
        time[window],
        plotted_lower[window],
        plotted_upper[window],
        color=ORANGE,
        alpha=0.5,
        linewidth=0,
        label="Spatial min–max",
    )
    ax.set_yscale("log")
    ax.set_xlim(time[window][0], time[window][-1])
    ax.set_title(title)
    ax.set_xlabel("Time (reduced units)")
    style_axis(ax)
axes_kl[0].set_ylabel("Local-equilibrium KL divergence")
axes_kl[0].legend(frameon=False)
fig_kl.suptitle("Kinetic neighborhood · nonequilibrium KL relaxation")
plt.show()